# Does the fitted root-n score vanish?
**Post-analysis only: loads saved results; no fitting or simulation.** Default: newest completed main CSV currently available, `main_26a7078945fc` (50 replications per sample size).

With $\widehat\zeta=\widehat\theta-\theta_0$, $\widetilde X=X-\varphi^*(Z)$ and $\widehat h=\widehat m-m_0+\widehat\zeta\varphi^*$,
$$S_n=\sqrt n\,\Psi_n(\widehat\zeta,\widehat h)
=-\frac1{\sqrt n}\sum_i\{\tau-\mathbf1(\widehat r_i<0)\}\widetilde X_i,\qquad
\widehat r_i=Y_i-\widehat\theta X_i-\widehat m(Z_i)
=\varepsilon_i-\widehat\zeta\widetilde X_i-\widehat h(Z_i).$$
Here $n$ is the actual fitting sample size (no 80/20 correction). The CSV already stores $S_n$ as **`S`**.

**Reparameterization matters:** holding $\widehat h$ fixed gives this residualized score. The original coefficient score holds $\widehat m$ fixed and uses $X$, so the two are not interchangeable:
$$S_n=\underbrace{-\sqrt n\,P_n[(\tau-\mathbf1\{\widehat r<0\})X]}_{\texttt{root_n_G}}
+\underbrace{\sqrt n\,P_n[(\tau-\mathbf1\{\widehat r<0\})\varphi^*(Z)]}_{\texttt{root_n_phi_score}}.$$
Polishing here continues joint fitting; it does not enforce exact minimization with $\widehat h$ frozen.
The plot is saved as `psi_score_trend.png` and `.pdf` in this main run's `figures/` directory.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RUN_NAME = "main_26a7078945fc"  # Change only to inspect another saved run.
THRESHOLDS = (0.02, 0.05, 0.10)  # Fixed thresholds across n.

candidates = [p / rel for p in [Path.cwd(), *Path.cwd().parents]
              for rel in (Path('.'), Path('results/2026-09-23_psi_n_thm33'))]
BASE = next((p for p in candidates if (p / 'run_expansion_v2' / RUN_NAME).is_dir()), None)
if BASE is None:
    raise FileNotFoundError('Open from this experiment folder or the dplqr repository.')
RUN = BASE / 'run_expansion_v2' / RUN_NAME
config = json.loads((RUN / 'manifest.json').read_text())['config']
df = pd.read_csv(RUN / 'replication_results.csv')
required = ['n', 'rep', 'stage', 'S', 'Psi_hat', 'root_n_G', 'root_n_phi_score']
if set(required) - set(df.columns):
    raise ValueError('Saved CSV is missing required score columns.')
if df.duplicated(['n', 'rep', 'stage']).any() or not np.isfinite(df[required[3:]].to_numpy()).all():
    raise ValueError('Duplicate replications or nonfinite scores; inspect the saved run.')
expected = pd.MultiIndex.from_product(
    [config['n_values'], range(1, config['reps'] + 1), ['baseline', 'polished']],
    names=['n', 'rep', 'stage'])
actual = pd.MultiIndex.from_frame(df[['n', 'rep', 'stage']])
if len(expected.difference(actual)) or len(actual.difference(expected)):
    raise ValueError('CSV does not contain the complete replication grid in the manifest.')
np.testing.assert_allclose(df.S, np.sqrt(df.n) * df.Psi_hat, atol=1e-10)
np.testing.assert_allclose(df.S, df.root_n_G + df.root_n_phi_score, atol=1e-10)
print('Source:', RUN.resolve())
print(f"Profile={config['profile']}; Q={config['reps']}; "
      f"epochs={config['baseline_epochs']} + {config['polish_epochs']} polishing")


In [ ]:
rows = []
for (stage, n), g in df.groupby(['stage', 'n'], sort=True):
    s = g.S.to_numpy()
    a = np.abs(s)
    rows.append(dict(stage=stage, n=n, Q=len(s), mean=s.mean(),
                     mean_abs=a.mean(), abs_MCSE=a.std(ddof=1)/np.sqrt(len(s)),
                     RMS=np.sqrt(np.mean(s*s)), q90_abs=np.quantile(a, .90),
                     **{f'Pr_abs_gt_{e:g}': np.mean(a > e) for e in THRESHOLDS},
                     mean_abs_G=g.root_n_G.abs().mean(),
                     mean_abs_phi=g.root_n_phi_score.abs().mean()))
summary = pd.DataFrame(rows)
display(summary.round(4))

# Descriptive log-log slopes: negative = decreasing; zero = flat.
# No asymptotic-rate claim or extrapolation from this small n grid.
trends = []
for stage, g in summary.groupby('stage'):
    g = g.sort_values('n')
    for metric in ['mean_abs', 'RMS', 'q90_abs']:
        y = g[metric].to_numpy()
        slope = np.polyfit(np.log(g.n), np.log(y), 1)[0] if len(y) > 1 and (y > 0).all() else np.nan
        trends.append(dict(stage=stage, metric=metric, log_log_slope=slope,
                           last_over_first=y[-1]/y[0] if y[0] > 0 else np.nan))
display(pd.DataFrame(trends).round(3))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.7), constrained_layout=True)
for stage, g in summary.groupby('stage'):
    g = g.sort_values('n')
    line = axes[0].errorbar(g.n, g.mean_abs, yerr=g.abs_MCSE,
                           marker='o', capsize=3, label=stage)
    color = line.lines[0].get_color()
    axes[1].plot(g.n, g.RMS, 'o-', color=color, label=stage)
    for e, style in zip(THRESHOLDS, ['-', '--', ':']):
        axes[2].plot(g.n, g[f'Pr_abs_gt_{e:g}'], marker='o', linestyle=style,
                     color=color, label=f'{stage}: epsilon={e:g}')
for ax, title in zip(axes, [r'Mean $|S_n|$ ($\pm1$ MCSE)',
                            r'RMS of $S_n$', r'Empirical $P(|S_n|>\epsilon)$']):
    ax.set(title=title, xlabel='Fitting sample size n')
    ax.set_xticks(sorted(summary.n.unique()))
    ax.set_ylim(bottom=0)
    ax.grid(alpha=.25)
    ax.legend(fontsize=8)
axes[2].set_ylim(-.02, 1.02)
fig.suptitle(RUN.name + r': $S_n=\sqrt{n}\,\Psi_n(\widehat{\zeta},\widehat{h})$')
FIG_DIR = RUN / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
for ext in ('png', 'pdf'):
    target = FIG_DIR / f'psi_score_trend.{ext}'
    fig.savefig(target, dpi=200, bbox_inches='tight')
    print('Saved:', target.resolve())
plt.show()


**Read the result:** $S_n=o_p(1)$ means $P(|S_n|>\epsilon)\to0$ for **every fixed** $\epsilon>0$. Decreasing absolute magnitudes and tail probabilities support this empirically; a plateau gives no evidence of vanishing over the tested range. A signed mean near zero can hide a nonvanishing spread. Mean absolute value/RMS assess moments, while the tails address convergence in probability directly (only at the displayed thresholds).

Use `mean_abs_G` and `mean_abs_phi` to see whether a small original coefficient score coexists with a substantial reparameterized contribution. Do not add these absolute means: the identity is for signed values, verified above. Neither a finite-grid slope nor these plots proves $o_p(1)$ or a nonvanishing $O_p(1)$ limit. This analysis uses the empirical score and needs no population integration.